# 00 - Colab Runner (data pipeline + demand model)

Runs the heavy compute on Colab: raw data download, preprocessing, and demand model training.
RL experiment runs get their own notebook (`03_rl_training.ipynb`).

Before running:
1. Runtime -> Change runtime type -> GPU (any GPU works, the MLP is small).
2. Have the repo in your Drive at `MyDrive/marl-dqn-dynamic-pricing`, or edit `REPO_PATH` below.
3. Runtime -> Run all. Everything is idempotent, re-running skips completed downloads.

Artifacts written back to the Drive repo: `data/processed/` and `results/demand/`.

In [ ]:
REPO_PATH = '/content/drive/MyDrive/marl-dqn-dynamic-pricing'

from google.colab import drive
drive.mount('/content/drive')
%cd {REPO_PATH}
%pip install -q -r requirements.txt
%pip install -q -e .

import torch
print('CUDA available:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU only)')

## 1. Raw data (downloads only what is missing, about 260MB on the first run)

In [ ]:
!python3 scripts/download_data.py

## 2. Preprocess on the fast local disk

The Drive mount is slow for heavy IO, so raw data is copied to the Colab local disk,
preprocessing runs there, and the processed artifacts are copied back to the Drive repo.

In [ ]:
import os

!mkdir -p /content/data
!cp -ru data/raw /content/data/
os.environ['AIRBNB_MARL_DATA_DIR'] = '/content/data'

!python3 scripts/preprocess.py

!mkdir -p data/processed && cp -r /content/data/processed/* data/processed/
!ls -lh data/processed/

## 3. Train the demand models (GPU)

In [ ]:
!AIRBNB_MARL_DATA_DIR=/content/data python3 scripts/train_demand.py

## 4. Summary

In [ ]:
import json

metrics = json.loads(open('results/demand/metrics.json').read())
print('=== TEST metrics ===')
for name, m in metrics['test'].items():
    print(f"  {name:15s} log-loss {m['log_loss']:.4f}  PR-AUC {m['pr_auc']:.4f}  "
          f"Brier {m['brier']:.4f}  ECE {m['ece']:.4f}")
print('\nelasticity correction:', json.dumps(metrics['elasticity_correction']))
print('\n=== Gates ===')
print(json.dumps(metrics['gates'], indent=2))
print('\nLR price_ratio coefficient:', metrics['lr_standardized_coefficients']['price_ratio'])

## Done

If all three gates pass, `results/demand/` in the Drive repo is the world model for the RL phases.
Sync the repo locally or read the summary above.